# 01 · Dataset Verification & Data Quality Audit

**Project:** Olist E-Commerce Customer Segmentation & Retention Analysis  
**Pipeline:** Notebook 1 of 5 · Next → [02 · Python Data Preparation](02_python_data_preparation.ipynb)

## Purpose
Before any modelling, this notebook verifies that the Olist Brazilian E-Commerce dataset (9 CSV tables from Kaggle) is complete and internally consistent. It checks file integrity, primary and foreign keys, missing values, data types, value ranges, and the logical order of order-lifecycle timestamps.

| | |
|---|---|
| **Input** | Original Kaggle ZIP archive (uploaded when prompted) |
| **Output** | Data-quality findings (the ZIP is extracted to `/content/olist_dataset`) |
| **Libraries** | pandas, csv, zipfile |
| **Environment** | Google Colab |

> **How to run:** execute the cells in order. The first cell opens an upload dialog; select the original Kaggle ZIP archive.

## 1. Setup: Load the Dataset
Upload the original Kaggle ZIP and extract the CSV files.

In [2]:

import os
import zipfile
import pandas as pd
from google.colab import files

# Upload the original Kaggle ZIP
uploaded = files.upload()

zip_path = next(iter(uploaded.keys()))

# Extract dataset
extract_path = "/content/olist_dataset"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print("Location:", extract_path)


Saving archive (5).zip to archive (5).zip
Dataset extracted successfully!
Location: /content/olist_dataset


### 1.1 File and row-count verification
Confirms that all nine expected files exist and reports rows, columns and size for each.

In [3]:

import os
import csv

expected_files = [
    "olist_customers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv"
]

# Locate extracted CSV files
available_files = {}

for root, dirs, filenames in os.walk(extract_path):
    for filename in filenames:
        if filename.endswith(".csv"):
            available_files[filename] = os.path.join(root, filename)

print("DATASET VERIFICATION REPORT")
print("=" * 65)

for filename in expected_files:
    if filename in available_files:
        path = available_files[filename]

        with open(path, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.reader(f)
            header = next(reader, [])
            row_count = sum(1 for _ in reader)

        size_mb = os.path.getsize(path) / (1024 ** 2)

        print(f"\nFile: {filename}")
        print(f"Rows: {row_count:,}")
        print(f"Columns: {len(header)}")
        print(f"Size: {size_mb:.2f} MB")
        print(f"Status: FOUND")
    else:
        print(f"\nFile: {filename}")
        print("Status: MISSING")


DATASET VERIFICATION REPORT

File: olist_customers_dataset.csv
Rows: 99,441
Columns: 5
Size: 8.62 MB
Status: FOUND

File: olist_orders_dataset.csv
Rows: 99,441
Columns: 8
Size: 16.84 MB
Status: FOUND

File: olist_order_items_dataset.csv
Rows: 112,650
Columns: 7
Size: 14.72 MB
Status: FOUND

File: olist_order_payments_dataset.csv
Rows: 103,886
Columns: 5
Size: 5.51 MB
Status: FOUND

File: olist_order_reviews_dataset.csv
Rows: 99,224
Columns: 7
Size: 13.78 MB
Status: FOUND

File: olist_products_dataset.csv
Rows: 32,951
Columns: 9
Size: 2.27 MB
Status: FOUND

File: olist_sellers_dataset.csv
Rows: 3,095
Columns: 4
Size: 0.17 MB
Status: FOUND

File: olist_geolocation_dataset.csv
Rows: 1,000,163
Columns: 5
Size: 58.44 MB
Status: FOUND

File: product_category_name_translation.csv
Rows: 71
Columns: 2
Size: 0.00 MB
Status: FOUND


#### Findings: File inventory
All 9 expected files were found.

| Table | Rows | Columns |
|---|---:|---:|
| customers | 99,441 | 5 |
| orders | 99,441 | 8 |
| order_items | 112,650 | 7 |
| order_payments | 103,886 | 5 |
| order_reviews | 99,224 | 7 |
| products | 32,951 | 9 |
| sellers | 3,095 | 4 |
| geolocation | 1,000,163 | 5 |
| product_category_name_translation | 71 | 2 |

## 2. Inspecting Dataset Structure

In [4]:

import pandas as pd
import os

print("TABLE SCHEMA INSPECTION")
print("=" * 70)

for filename in expected_files:

    path = available_files[filename]

    # Read only 5 rows to inspect the structure
    df_sample = pd.read_csv(path, nrows=5)

    print(f"\n{'=' * 70}")
    print("TABLE:", filename)
    print("COLUMNS:", list(df_sample.columns))
    print("\nSAMPLE DATA:")
    display(df_sample.head(3))


TABLE SCHEMA INSPECTION

TABLE: olist_customers_dataset.csv
COLUMNS: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

SAMPLE DATA:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



TABLE: olist_orders_dataset.csv
COLUMNS: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

SAMPLE DATA:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



TABLE: olist_order_items_dataset.csv
COLUMNS: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

SAMPLE DATA:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


... (8 outputs left out)


TABLE: olist_geolocation_dataset.csv
COLUMNS: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

SAMPLE DATA:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP



TABLE: product_category_name_translation.csv
COLUMNS: ['product_category_name', 'product_category_name_english']

SAMPLE DATA:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


## 3. Primary Key and Data Quality Checks

### 3.1 Candidate primary keys
Checks every table for duplicate or null values in its candidate key.

In [5]:

import pandas as pd
import os

key_checks = {
    "olist_customers_dataset.csv": ["customer_id"],
    "olist_orders_dataset.csv": ["order_id"],
    "olist_order_items_dataset.csv": ["order_id", "order_item_id"],
    "olist_order_payments_dataset.csv": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset.csv": ["review_id"],
    "olist_products_dataset.csv": ["product_id"],
    "olist_sellers_dataset.csv": ["seller_id"],
    "product_category_name_translation.csv": [
        "product_category_name"
    ]
}

for filename, keys in key_checks.items():

    df = pd.read_csv(available_files[filename])

    print("\nTABLE:", filename)

    duplicate_count = df.duplicated(subset=keys).sum()
    null_count = df[keys].isnull().any(axis=1).sum()

    print("Candidate key:", keys)
    print("Duplicate key rows:", duplicate_count)
    print("Null key rows:", null_count)

    if duplicate_count == 0 and null_count == 0:
        print("Result: Candidate key passed")
    else:
        print("Result: Needs investigation")



TABLE: olist_customers_dataset.csv
Candidate key: ['customer_id']
Duplicate key rows: 0
Null key rows: 0
Result: Candidate key passed

TABLE: olist_orders_dataset.csv
Candidate key: ['order_id']
Duplicate key rows: 0
Null key rows: 0
Result: Candidate key passed

TABLE: olist_order_items_dataset.csv
Candidate key: ['order_id', 'order_item_id']
Duplicate key rows: 0
Null key rows: 0
Result: Candidate key passed

TABLE: olist_order_payments_dataset.csv
Candidate key: ['order_id', 'payment_sequential']
Duplicate key rows: 0
Null key rows: 0
Result: Candidate key passed

TABLE: olist_order_reviews_dataset.csv
Candidate key: ['review_id']
Duplicate key rows: 814
Null key rows: 0
Result: Needs investigation

TABLE: olist_products_dataset.csv
Candidate key: ['product_id']
Duplicate key rows: 0
Null key rows: 0
Result: Candidate key passed

TABLE: olist_sellers_dataset.csv
Candidate key: ['seller_id']
Duplicate key rows: 0
Null key rows: 0
Result: Candidate key passed

TABLE: product_category

#### Findings: Primary keys
7 of 8 tables pass their candidate-key check. The exception is **`order_reviews`**: `review_id` has 814 duplicate key rows, which is investigated next.

### 3.2 Investigation: duplicate `review_id` values
Are the repeated review IDs exact duplicates, or do they represent something else?

In [6]:
reviews_path = available_files["olist_order_reviews_dataset.csv"]

reviews = pd.read_csv(reviews_path)

# Find every row whose review_id appears more than once
duplicate_reviews = reviews[
    reviews.duplicated(subset=["review_id"], keep=False)
].sort_values("review_id")

print("Total review rows:", len(reviews))
print("Unique review IDs:", reviews["review_id"].nunique())
print("Rows involved in duplicate IDs:", len(duplicate_reviews))

display(duplicate_reviews.head(20))

Total review rows: 99224
Unique review IDs: 98410
Rows involved in duplicate IDs: 1603


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [7]:
# Check whether the duplicated rows are completely identical
exact_duplicates = reviews.duplicated().sum()

# Count how many review IDs occur more than once
duplicate_id_count = (
    reviews["review_id"].value_counts().gt(1).sum()
)

print("Exact duplicate rows:", exact_duplicates)
print("Review IDs appearing multiple times:", duplicate_id_count)

Exact duplicate rows: 0
Review IDs appearing multiple times: 789


In [8]:
# Exclude order_id because we already know it can differ
review_info_columns = [
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
]

review_id_analysis = (
    reviews.groupby("review_id")
    .agg(
        order_count=("order_id", "nunique"),
        score_count=("review_score", "nunique"),
        message_count=("review_comment_message", "nunique"),
        date_count=("review_creation_date", "nunique")
    )
)

print("Review IDs linked to multiple orders:",
      (review_id_analysis["order_count"] > 1).sum())

print("Repeated IDs with different scores:",
      ((review_id_analysis["order_count"] > 1) &
       (review_id_analysis["score_count"] > 1)).sum())

print("Repeated IDs with different messages:",
      ((review_id_analysis["order_count"] > 1) &
       (review_id_analysis["message_count"] > 1)).sum())

print("Repeated IDs with different creation dates:",
      ((review_id_analysis["order_count"] > 1) &
       (review_id_analysis["date_count"] > 1)).sum())

Review IDs linked to multiple orders: 789
Repeated IDs with different scores: 0
Repeated IDs with different messages: 0
Repeated IDs with different creation dates: 0


In [9]:
print("Duplicate order_id rows in reviews:",
      reviews.duplicated(subset=["order_id"]).sum())

print("Null order_id rows:",
      reviews["order_id"].isnull().sum())

Duplicate order_id rows in reviews: 551
Null order_id rows: 0


In [10]:
# Find orders that have more than one review record
duplicate_order_reviews = (
    reviews[reviews.duplicated(subset=["order_id"], keep=False)]
    .sort_values("order_id")
)

print("Rows belonging to repeated order IDs:",
      len(duplicate_order_reviews))

print("Number of affected orders:",
      duplicate_order_reviews["order_id"].nunique())

display(duplicate_order_reviews.head(20))

Rows belonging to repeated order IDs: 1098
Number of affected orders: 547


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25 00:00:00,2017-08-29 21:45:57
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:47
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29 00:00:00,2018-03-30 03:16:19
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21 00:00:00,2018-03-22 01:32:08
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19 00:00:00,2017-07-20 12:06:11
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14 00:00:00,2017-07-17 13:58:06


In [11]:
all_review_columns = [
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
]

repeated_ids = reviews[
    reviews.duplicated(subset=["review_id"], keep=False)
]

differences = (
    repeated_ids.groupby("review_id")[all_review_columns]
    .nunique(dropna=False)
)

print(
    "Repeated review IDs with differences in any review field:",
    (differences > 1).any(axis=1).sum()
)

Repeated review IDs with differences in any review field: 0


In [12]:
orders = pd.read_csv(
    available_files["olist_orders_dataset.csv"],
    usecols=["order_id"]
)

unmatched_reviews = reviews[
    ~reviews["order_id"].isin(orders["order_id"])
]

print("Reviews with no matching order:",
      len(unmatched_reviews))

Reviews with no matching order: 0


In [13]:
print(
    "Duplicate review_id + order_id combinations:",
    reviews.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

print(
    "Null review_id:",
    reviews["review_id"].isnull().sum()
)

print(
    "Null order_id:",
    reviews["order_id"].isnull().sum()
)

Duplicate review_id + order_id combinations: 0
Null review_id: 0
Null order_id: 0


In [14]:
reviews_per_order = (
    reviews.groupby("order_id")
    .size()
    .value_counts()
    .sort_index()
)

print("Distribution of reviews per order:")
display(reviews_per_order)

print(
    "Maximum reviews for a single order:",
    reviews.groupby("order_id").size().max()
)

Distribution of reviews per order:


,count
1,98126
2,543
3,4


Maximum reviews for a single order: 3


#### Findings: Duplicate review IDs
- The reviews table has 99,224 rows but only 98,410 unique `review_id` values; **789 review IDs** appear more than once (1,603 rows).
- There are **no fully duplicated rows**. For every repeated `review_id`, the score, comment text and dates are identical and only the `order_id` differs, so the same review is attached to more than one order.
- 547 orders have more than one review record (543 orders with 2 reviews, 4 orders with 3).
- Every review links to an existing order (0 orphan reviews), and the combination `review_id` + `order_id` is unique.

**Conclusion:** `review_id` alone is not a valid primary key for `order_reviews`; the (`review_id`, `order_id`) pair is.

### 3.3 Foreign-key (referential integrity) checks
Verifies that every child record points to an existing parent record.

In [15]:
# Load parent tables
customers = pd.read_csv(available_files["olist_customers_dataset.csv"])
orders = pd.read_csv(available_files["olist_orders_dataset.csv"])
products = pd.read_csv(available_files["olist_products_dataset.csv"])
sellers = pd.read_csv(available_files["olist_sellers_dataset.csv"])

# Load child tables
items = pd.read_csv(available_files["olist_order_items_dataset.csv"])
payments = pd.read_csv(available_files["olist_order_payments_dataset.csv"])

# Define relationships
foreign_key_checks = [
    (orders, "customer_id", customers, "customer_id", "Orders -> Customers"),
    (items, "order_id", orders, "order_id", "Items -> Orders"),
    (items, "product_id", products, "product_id", "Items -> Products"),
    (items, "seller_id", sellers, "seller_id", "Items -> Sellers"),
    (payments, "order_id", orders, "order_id", "Payments -> Orders"),
    (reviews, "order_id", orders, "order_id", "Reviews -> Orders")
]

for child, child_col, parent, parent_col, name in foreign_key_checks:

    unmatched = ~child[child_col].isin(parent[parent_col])

    print("\nRelationship:", name)
    print("Total child records:", len(child))
    print("Unmatched foreign keys:", unmatched.sum())

    if unmatched.sum() == 0:
        print("Result: PASSED")
    else:
        print("Result: INVESTIGATE")


Relationship: Orders -> Customers
Total child records: 99441
Unmatched foreign keys: 0
Result: PASSED

Relationship: Items -> Orders
Total child records: 112650
Unmatched foreign keys: 0
Result: PASSED

Relationship: Items -> Products
Total child records: 112650
Unmatched foreign keys: 0
Result: PASSED

Relationship: Items -> Sellers
Total child records: 112650
Unmatched foreign keys: 0
Result: PASSED

Relationship: Payments -> Orders
Total child records: 103886
Unmatched foreign keys: 0
Result: PASSED

Relationship: Reviews -> Orders
Total child records: 99224
Unmatched foreign keys: 0
Result: PASSED


#### Findings: Referential integrity
All 6 relationships pass with **0 unmatched foreign keys**: orders → customers, items → orders / products / sellers, payments → orders, reviews → orders.

## 4. Missing Values
Counts missing values per column and examines whether the gaps are explainable.

In [16]:
import pandas as pd

missing_report = []

for filename in expected_files:

    df = pd.read_csv(available_files[filename])

    for column in df.columns:

        missing_count = df[column].isnull().sum()

        missing_report.append({
            "table": filename,
            "column": column,
            "missing_count": missing_count,
            "missing_percentage": round(
                (missing_count / len(df)) * 100, 2
            )
        })

missing_df = pd.DataFrame(missing_report)

missing_df = missing_df[
    missing_df["missing_count"] > 0
].sort_values(
    by="missing_percentage",
    ascending=False
)

print("Columns containing missing values:", len(missing_df))

display(missing_df)

Columns containing missing values: 13


,table,column,missing_count,missing_percentage
28,olist_order_reviews_dataset.csv,review_comment_title,87656,88.34
29,olist_order_reviews_dataset.csv,review_comment_message,58247,58.70
11,olist_orders_dataset.csv,order_delivered_customer_date,2965,2.98
34,olist_products_dataset.csv,product_name_lenght,610,1.85
33,olist_products_dataset.csv,product_category_name,610,1.85
35,olist_products_dataset.csv,product_description_lenght,610,1.85
36,olist_products_dataset.csv,product_photos_qty,610,1.85
10,olist_orders_dataset.csv,order_delivered_carrier_date,1783,1.79
9,olist_orders_dataset.csv,order_approved_at,160,0.16
37,olist_products_dataset.csv,product_weight_g,2,0.01


In [17]:
orders["order_status"] = orders["order_status"].fillna("Unknown")

date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for column in date_columns:
    print("\nMissing:", column)
    display(
        orders.loc[
            orders[column].isna(), "order_status"
        ].value_counts()
    )


Missing: order_approved_at


,count
order_status,
canceled,141
delivered,14
created,5



Missing: order_delivered_carrier_date


,count
order_status,
unavailable,609
canceled,550
invoiced,314
processing,301
created,5
approved,2
delivered,2



Missing: order_delivered_customer_date


,count
order_status,
shipped,1107
canceled,619
unavailable,609
invoiced,314
processing,301
delivered,8
created,5
approved,2


In [18]:
product_missing_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

product_missing_pattern = (
    products[product_missing_columns]
    .isna()
    .sum(axis=1)
    .value_counts()
    .sort_index()
)

print("Missing fields per product:")
display(product_missing_pattern)

print(
    "\nRows where the first four fields are all missing:",
    products[
        [
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty"
        ]
    ].isna().all(axis=1).sum()
)

print(
    "\nRows with any missing dimension:",
    products[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isna().any(axis=1).sum()
)

Missing fields per product:


,count
0,32340
4,610
8,1



Rows where the first four fields are all missing: 610

Rows with any missing dimension: 2


In [19]:
date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

delivered_anomalies = orders[
    (orders["order_status"] == "delivered") &
    (orders[date_columns].isna().any(axis=1))
]

print("Delivered orders with missing dates:", len(delivered_anomalies))

display(
    delivered_anomalies[
        ["order_id", "order_status"] + date_columns
    ]
)

Delivered orders with missing dates: 23


,order_id,order_status,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN
22663,5cf925b116421afa85ee25e99b4c34fb,delivered,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47
23156,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19
26800,c1d4211b3dae76144deccd6c74144a88,delivered,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01
38290,d69e5d356402adc8cf17e08b5033acfb,delivered,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58
39334,d77031d6a3c8a52f019764e68f211c69,delivered,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23


In [20]:
product_missing_details = products[
    products[product_missing_columns].isna().any(axis=1)
]

print("Products with missing information:", len(product_missing_details))

display(
    product_missing_details[
        ["product_id"] + product_missing_columns
    ]
)

Products with missing information: 611


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


#### Findings: Missing values
- **13 columns** across 3 tables contain missing values.
- **Reviews:** `review_comment_title` is missing in 88.34% of rows and `review_comment_message` in 58.70%. These are optional free-text fields.
- **Orders:** `order_delivered_customer_date` is missing for 2,965 orders (2.98%), `order_delivered_carrier_date` for 1,783 (1.79%) and `order_approved_at` for 160 (0.16%). Most gaps coincide with orders that had not reached delivery (shipped, canceled, unavailable, invoiced, processing).
- **23 orders marked `delivered` still have at least one missing lifecycle date**; these are the records to treat with caution.
- **Products:** 611 products have missing information. 610 of them lack all four descriptive fields (category, name length, description length, photo count). Only 2 products are missing weight/dimensions.

## 5. Data Types and Date Validation
Checks that every date column can be parsed into a proper datetime.

In [21]:
dtype_report = []

for filename in expected_files:
    df = pd.read_csv(available_files[filename], nrows=1000)

    for column in df.columns:
        dtype_report.append({
            "table": filename,
            "column": column,
            "dtype": str(df[column].dtype)
        })

dtype_df = pd.DataFrame(dtype_report)

display(dtype_df)

,table,column,dtype
0,olist_customers_dataset.csv,customer_id,object
1,olist_customers_dataset.csv,customer_unique_id,object
2,olist_customers_dataset.csv,customer_zip_code_prefix,int64
3,olist_customers_dataset.csv,customer_city,object
4,olist_customers_dataset.csv,customer_state,object
5,olist_orders_dataset.csv,order_id,object
6,olist_orders_dataset.csv,customer_id,object
7,olist_orders_dataset.csv,order_status,object
8,olist_orders_dataset.csv,order_purchase_timestamp,object
9,olist_orders_dataset.csv,order_approved_at,object


In [22]:
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ],
    "items": ["shipping_limit_date"]
}

for table_name, columns in date_columns.items():
    df = globals()[table_name]

    print(f"\n--- {table_name.upper()} ---")

    for column in columns:
        converted = pd.to_datetime(df[column], errors="coerce")

        invalid = df[column].notna() & converted.isna()

        print(
            column,
            "| Original dtype:", df[column].dtype,
            "| Invalid dates:", invalid.sum()
        )


--- ORDERS ---
order_purchase_timestamp | Original dtype: object | Invalid dates: 0
order_approved_at | Original dtype: object | Invalid dates: 0
order_delivered_carrier_date | Original dtype: object | Invalid dates: 0
order_delivered_customer_date | Original dtype: object | Invalid dates: 0
order_estimated_delivery_date | Original dtype: object | Invalid dates: 0

--- REVIEWS ---
review_creation_date | Original dtype: object | Invalid dates: 0
review_answer_timestamp | Original dtype: object | Invalid dates: 0

--- ITEMS ---
shipping_limit_date | Original dtype: object | Invalid dates: 0


#### Findings: Data types
All date columns are read as text (`object`) and must be converted explicitly. Every date column converts cleanly, with **0 invalid dates**.

## 6. Numeric Value Sanity Checks
Looks for impossible values (negative amounts, scores outside 1–5, invalid coordinates, zero weights or installments).

In [23]:
# Check numeric columns for suspicious values

checks = {
    "order_items": {
        "price": lambda s: s < 0,
        "freight_value": lambda s: s < 0,
        "order_item_id": lambda s: s <= 0
    },
    "payments": {
        "payment_value": lambda s: s < 0,
        "payment_installments": lambda s: s <= 0,
        "payment_sequential": lambda s: s <= 0
    },
    "reviews": {
        "review_score": lambda s: ~s.between(1, 5)
    },
    "products": {
        "product_weight_g": lambda s: s <= 0,
        "product_length_cm": lambda s: s <= 0,
        "product_height_cm": lambda s: s <= 0,
        "product_width_cm": lambda s: s <= 0
    },
    "geolocation": {
        "geolocation_lat": lambda s: ~s.between(-90, 90),
        "geolocation_lng": lambda s: ~s.between(-180, 180)
    }
}

tables = {
    "order_items": items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "geolocation": pd.read_csv(
        available_files["olist_geolocation_dataset.csv"]
    )
}

anomaly_report = []

for table_name, column_checks in checks.items():
    df = tables[table_name]

    for column, condition in column_checks.items():
        invalid_mask = df[column].notna() & condition(df[column])

        anomaly_report.append({
            "table": table_name,
            "column": column,
            "anomaly_count": int(invalid_mask.sum())
        })

anomaly_df = pd.DataFrame(anomaly_report)
display(anomaly_df)

,table,column,anomaly_count
0,order_items,price,0
1,order_items,freight_value,0
2,order_items,order_item_id,0
3,payments,payment_value,0
4,payments,payment_installments,2
5,payments,payment_sequential,0
6,reviews,review_score,0
7,products,product_weight_g,4
8,products,product_length_cm,0
9,products,product_height_cm,0


In [24]:
payment_anomalies = payments[
    payments["payment_installments"] <= 0
]

print("Anomalous payment records:", len(payment_anomalies))

display(payment_anomalies)

Anomalous payment records: 2


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [25]:
weight_anomalies = products[
    products["product_weight_g"] <= 0
]

print("Anomalous product records:", len(weight_anomalies))

display(
    weight_anomalies[
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ]
)

Anomalous product records: 4


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,0.0,30.0,25.0,30.0


#### Findings: Numeric anomalies
- No negative prices, freight values or payment values; all review scores are within 1–5; all coordinates are valid.
- **2 payment records** have `payment_installments = 0` (both credit-card payments with `payment_sequential = 2`).
- **4 products** have a weight of 0 g (all in the `cama_mesa_banho` category).

## 7. Timeline Consistency of Order Events
Checks that order events happen in a logical sequence: purchase → approval → carrier hand-off → customer delivery.

In [26]:
date_sequence = [
    ("order_purchase_timestamp", "order_approved_at"),
    ("order_approved_at", "order_delivered_carrier_date"),
    ("order_delivered_carrier_date", "order_delivered_customer_date")
]

orders_dates = orders.copy()

for column in orders_dates.columns:
    if column.endswith("_timestamp") or column.endswith("_date"):
        orders_dates[column] = pd.to_datetime(
            orders_dates[column], errors="coerce"
        )

for earlier, later in date_sequence:
    valid = (
        orders_dates[earlier].notna() &
        orders_dates[later].notna()
    )

    violations = orders_dates.loc[
        valid & (orders_dates[earlier] > orders_dates[later]),
        ["order_id", "order_status", earlier, later]
    ]

    print(f"\n{earlier} > {later}")
    print("Chronological violations:", len(violations))

    display(violations.head(10))


order_purchase_timestamp > order_approved_at
Chronological violations: 0


,order_id,order_status,order_purchase_timestamp,order_approved_at



order_approved_at > order_delivered_carrier_date
Chronological violations: 1359


,order_id,order_status,order_approved_at,order_delivered_carrier_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-12 23:31:02,2018-06-11 14:54:00
64,688052146432ef8253587b930b01a06d,delivered,2018-04-24 18:25:22,2018-04-23 19:19:14
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-26 23:31:53,2018-07-24 12:57:00
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-23 12:31:53,2018-07-23 12:24:00
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-27 23:31:09,2018-07-24 14:03:00
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-05 16:33:06,2018-07-05 14:50:00
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-29 23:30:52,2018-07-26 14:46:00
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 16:52:39,2018-05-07 15:09:00
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 14:05:13,2018-08-16 13:27:00
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-05 16:31:26,2018-07-04 12:14:00



order_delivered_carrier_date > order_delivered_customer_date
Chronological violations: 23


,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-28 16:57:58,2017-07-25 19:32:56
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-07 17:22:41,2017-07-06 14:27:51
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-20 19:22:02,2017-07-19 14:13:28
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-20 23:03:42,2017-07-20 18:52:41
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-26 11:41:53,2016-10-25 17:51:46
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-27 14:51:54,2017-06-26 15:45:35
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-10 18:28:56,2017-08-10 18:05:38
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-08-01 18:17:47,2017-07-31 17:49:56
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-08-09 18:18:43,2017-08-01 21:13:01


In [27]:
orders_dates = orders.copy()

date_columns_to_check = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for col in date_columns_to_check:
    orders_dates[col] = pd.to_datetime(
        orders_dates[col], errors="coerce"
    )

mask = (
    orders_dates["order_approved_at"].notna() &
    orders_dates["order_delivered_carrier_date"].notna() &
    (
        orders_dates["order_approved_at"] >
        orders_dates["order_delivered_carrier_date"]
    )
)

approval_carrier_issues = orders_dates[mask].copy()

print("Total affected orders:", len(approval_carrier_issues))

print("\nAffected order statuses:")
display(approval_carrier_issues["order_status"].value_counts())

approval_carrier_issues["difference_hours"] = (
    approval_carrier_issues["order_approved_at"] -
    approval_carrier_issues["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

print("\nTime difference in hours:")
display(
    approval_carrier_issues["difference_hours"].describe()
)

print("\nEarliest affected records:")
display(
    approval_carrier_issues[
        [
            "order_id",
            "order_status",
            "order_approved_at",
            "order_delivered_carrier_date",
            "difference_hours"
        ]
    ].head(10)
)

Total affected orders: 1359

Affected order statuses:


,count
order_status,
delivered,1350
shipped,9



Time difference in hours:


,difference_hours
count,1359.000000
mean,24.751987
std,115.307793
min,0.005833
25%,1.415417
50%,17.167778
75%,25.956667
max,4109.256111



Earliest affected records:


,order_id,order_status,order_approved_at,order_delivered_carrier_date,difference_hours
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-12 23:31:02,2018-06-11 14:54:00,32.617222
64,688052146432ef8253587b930b01a06d,delivered,2018-04-24 18:25:22,2018-04-23 19:19:14,23.102222
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-26 23:31:53,2018-07-24 12:57:00,58.581389
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-23 12:31:53,2018-07-23 12:24:00,0.131389
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-27 23:31:09,2018-07-24 14:03:00,81.469167
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-05 16:33:06,2018-07-05 14:50:00,1.718333
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-29 23:30:52,2018-07-26 14:46:00,80.747778
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 16:52:39,2018-05-07 15:09:00,1.727500
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 14:05:13,2018-08-16 13:27:00,0.636944
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-05 16:31:26,2018-07-04 12:14:00,28.290556


In [28]:
mask_delivery = (
    orders_dates["order_delivered_carrier_date"].notna() &
    orders_dates["order_delivered_customer_date"].notna() &
    (
        orders_dates["order_delivered_carrier_date"] >
        orders_dates["order_delivered_customer_date"]
    )
)

delivery_issues = orders_dates[mask_delivery].copy()

delivery_issues["difference_hours"] = (
    delivery_issues["order_delivered_carrier_date"] -
    delivery_issues["order_delivered_customer_date"]
).dt.total_seconds() / 3600

print("Total affected orders:", len(delivery_issues))

print("\nOrder statuses:")
display(delivery_issues["order_status"].value_counts())

print("\nDiscrepancy statistics:")
display(delivery_issues["difference_hours"].describe())

print("\nAffected records:")
display(
    delivery_issues[
        [
            "order_id",
            "order_status",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "difference_hours"
        ]
    ]
)

Total affected orders: 23

Order statuses:


,count
order_status,
delivered,23



Discrepancy statistics:


,difference_hours
count,23.000000
mean,78.455133
std,89.308070
min,0.388333
25%,23.338333
50%,39.864444
75%,128.759028
max,386.308056



Affected records:


,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date,difference_hours
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-28 16:57:58,2017-07-25 19:32:56,69.417222
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-07 17:22:41,2017-07-06 14:27:51,26.913889
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-20 19:22:02,2017-07-19 14:13:28,29.142778
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10,144.238889
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-20 23:03:42,2017-07-20 18:52:41,4.183611
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-26 11:41:53,2016-10-25 17:51:46,17.835278
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-27 14:51:54,2017-06-26 15:45:35,23.105278
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-10 18:28:56,2017-08-10 18:05:38,0.388333
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-08-01 18:17:47,2017-07-31 17:49:56,24.464167
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-08-09 18:18:43,2017-08-01 21:13:01,189.095000


In [29]:
approval_carrier_ids = set(
    approval_carrier_issues["order_id"]
)

carrier_delivery_ids = set(
    delivery_issues["order_id"]
)

overlap_ids = approval_carrier_ids & carrier_delivery_ids

all_anomaly_ids = (
    approval_carrier_ids | carrier_delivery_ids
)

print("Approval/carrier anomalies:", len(approval_carrier_ids))
print("Carrier/delivery anomalies:", len(carrier_delivery_ids))
print("Orders appearing in both groups:", len(overlap_ids))
print("Total unique affected orders:", len(all_anomaly_ids))

print(
    "Percentage of all orders affected:",
    round(len(all_anomaly_ids) / len(orders) * 100, 2),
    "%"
)

Approval/carrier anomalies: 1359
Carrier/delivery anomalies: 23
Orders appearing in both groups: 0
Total unique affected orders: 1382
Percentage of all orders affected: 1.39 %


#### Findings: Timeline anomalies
- Purchase → approval: **0 violations**.
- Approval *after* carrier hand-off: **1,359 orders** (1,350 delivered, 9 shipped). The median gap is about 17 hours, with a maximum of roughly 4,109 hours.
- Carrier hand-off *after* customer delivery: **23 orders**, all delivered (median gap about 40 hours).
- The two groups do not overlap, giving **1,382 unique affected orders (1.39% of all orders)**. These records are flagged here, not removed.

## 8. Summary
| Check | Result |
|---|---|
| File completeness | 9 of 9 files present |
| Primary keys | 7 of 8 pass; `review_id` is repeated across orders (use `review_id` + `order_id`) |
| Referential integrity | 6 of 6 relationships pass |
| Missing values | 13 columns affected; mostly optional review text and orders not yet delivered |
| Data types | Dates stored as text; all parse successfully |
| Value ranges | 2 zero-installment payments, 4 zero-weight products |
| Timeline logic | 1,382 orders (1.39%) with out-of-sequence timestamps |

The dataset is structurally sound and suitable for analysis. Notebook 02 prepares the tables for SQL loading and validates the SQL-generated RFM table.